In [1]:
import pandas as pd
import numpy as np

TARGETS = ["y"]

In [2]:
results_1l = pd.read_excel("resultados-1l-v2.xlsx")
# results_2l = pd.read_excel("resultados-2l.xlsx")
# results_3l = pd.read_excel("resultados-3l.xlsx")

# results = pd.concat(
#     [results_1l, results_2l,],
#     ignore_index=True )
results = results_1l


In [3]:
results

,model,Neurons,Ld,Lp,reg,seed,R2_ZZx1_y,MSE_ZZx1_y,R2_ZZx2_y,MSE_ZZx2_y,...,R2_LSG_1_y,MSE_LSG_1_y,R2_LSG_2_y,MSE_LSG_2_y,R2_ZZx1_inv_y,MSE_ZZx1_inv_y,R2_zzx2_inv2_y,MSE_zzx2_inv2_y,R2_semiCirc_y,MSE_semiCirc_y
0,model_arch1_r0.01_Ld0.5_Lp0.5_seed9910,[1],0.5,0.5,0.01,9910,-1.856291,0.031055,-1.715586,0.022286,...,0.093571,0.047787,-0.742197,0.037792,-1.307931,0.002337,-1.397858,0.015641,-0.926723,-0.002578
1,model_arch1_r0.01_Ld0.5_Lp0.5_seed3211,[1],0.5,0.5,0.01,3211,-1.841470,0.037065,-1.699286,0.026324,...,0.100665,0.047501,-0.733251,0.045107,-1.250710,0.003368,-1.401530,0.018373,-0.926590,-0.002828
2,model_arch1_r0.01_Ld0.5_Lp0.5_seed8561,[1],0.5,0.5,0.01,8561,-1.918644,0.027801,-1.626363,0.024434,...,0.135861,0.085404,-0.877413,0.037511,-1.169109,-0.004745,-1.439113,0.016942,-0.829988,-0.008858
3,model_arch1_r0.01_Ld0.5_Lp0.5_seed2023,[1],0.5,0.5,0.01,2023,-1.983460,0.004433,-1.731923,0.005995,...,0.100303,0.083294,-0.952740,0.008634,-1.379060,-0.008451,-1.418669,0.005095,-0.832463,-0.006359
4,model_arch1_r0.01_Ld0.5_Lp0.5_seed6724,[1],0.5,0.5,0.01,6724,-2.020648,-0.006417,-1.764485,0.000699,...,0.093080,0.088646,-1.006322,-0.003829,-1.433817,-0.011749,-1.411601,0.002159,-0.926072,-0.008609
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1911,model_arch50_r0.01_Ld0.3_Lp0.7_seed7732,[50],0.3,0.7,0.01,7732,-2.129003,-0.014376,-1.390824,0.009971,...,0.237778,0.194395,-1.475464,0.002432,-0.933551,-0.059563,-1.663875,-0.012412,-0.316424,-0.034232
1912,model_arch50_r0.01_Ld0.3_Lp0.7_seed1657,[50],0.3,0.7,0.01,1657,-1.983048,0.027874,-1.303127,0.041714,...,0.266407,0.184850,-1.261529,0.049262,-0.738860,-0.051443,-1.724929,-0.005576,-0.159699,-0.035300
1913,model_arch50_r0.01_Ld0.3_Lp0.7_seed6995,[50],0.3,0.7,0.01,6995,-1.965159,-0.002795,-1.595153,0.000259,...,0.148685,0.127061,-1.119575,0.006119,-1.334522,-0.027134,-1.619469,-0.018266,-0.413689,-0.011894
1914,model_arch50_r0.9_Ld0.3_Lp0.7_seed639,[50],0.3,0.7,0.90,639,-2.217651,-0.046940,-1.279247,-0.004282,...,0.310115,0.258660,-1.897628,-0.023316,-0.745884,-0.119341,-1.871815,-0.044628,0.023047,-0.060193


In [6]:
# 🔹 categorização dos sets (baseada nos comentários originais)
SETS_CATEGORY = {
    "ZZx1":     "Train",
    "ZZx2":     "Val",
    "ZZxReto":  "Test",
    "ZZy1":     "Test",
    "ZZy2":     "Test",
    "LSG-1":    "Test",
    "LSG-2":    "Test",
    "ZZx1-inv": "Test",
    "ZZx2-inv": "Test",
    "semiCirc": "Test",
}

def col_name(s, target):
    # sanitiza "-" pra "_" pra bater com o nome real da coluna, se for o caso
    return f"R2_{s.replace('-', '_')}_{target}"

best_models_tables = {}
N = 5  # top modelos

w_val = 0.0
w_train = 0.33
w_test = 0.0


for target in TARGETS:

    # 🔹 sets de Train, Val e Test
    train_sets = [s for s, cat in SETS_CATEGORY.items() if cat == "Train"]
    val_sets   = [s for s, cat in SETS_CATEGORY.items() if cat == "Val"]
    test_sets  = [s for s, cat in SETS_CATEGORY.items() if cat == "Test"]

    train_cols = [col_name(s, target) for s in train_sets]
    val_cols   = [col_name(s, target) for s in val_sets]
    test_cols  = [col_name(s, target) for s in test_sets]

    # 🔹 garantir que só usamos colunas existentes
    train_cols = [c for c in train_cols if c in results.columns]
    val_cols   = [c for c in val_cols if c in results.columns]
    test_cols  = [c for c in test_cols if c in results.columns]

    r2_all_cols = train_cols + val_cols + test_cols

    if not r2_all_cols:
        print(f"⚠️ Nenhuma coluna Train/Val/Test encontrada para target={target}, pulando.")
        continue

    df = results.copy()

    # 🔹 remover linhas onde QUALQUER R2 (Train/Val/Test) < 0
    # df = df[(df[r2_all_cols] >= 0).all(axis=1)]

    # =========================
    # 🔹 MÉDIAS POR GRUPO
    # =========================
    df["R2_train_mean"] = df[train_cols].mean(axis=1) if train_cols else np.nan
    df["R2_val_mean"]   = df[val_cols].mean(axis=1) if val_cols else np.nan
    df["R2_test_mean"]  = df[test_cols].mean(axis=1) if test_cols else np.nan

    # =========================
    # 🔹 SCORE
    # =========================
    df["R2_std"] = df[r2_all_cols].std(axis=1)

    df["Score"] = (
        w_train * df["R2_train_mean"] +
        w_val   * df["R2_val_mean"] +
        w_test  * df["R2_test_mean"] - 
        0.1 * df["R2_std"]   # penaliza inconsistência
    )

    # =========================
    # 🔹 ORDENAÇÃO
    # =========================
    df_sorted = df.sort_values(by="Score", ascending=False)
    best_models_tables[target] = df_sorted

    # =========================
    # 🔹 TOP N RESUMO
    # =========================
    print(f"\n🏆 TOP {N} MODELOS - {target}")
    display(df_sorted[
        ["model", "Neurons", "R2_train_mean", "R2_val_mean", "R2_test_mean", "Score"]
    ].head(N))

    top_df = df_sorted.head(N).copy()

    final_cols = ["model", "Neurons"] + r2_all_cols + [
        "R2_train_mean", "R2_val_mean", "R2_test_mean", "Score"
    ]
    final_table = top_df[final_cols]

    print(f"\n📊 MÉTRICAS COMPLETAS - TOP {N} ({target})")
    display(final_table)


🏆 TOP 5 MODELOS - y


,model,Neurons,R2_train_mean,R2_val_mean,R2_test_mean,Score
22,model_arch1_r0.01_Ld0.7_Lp0.3_seed8561,[1],-0.177143,-0.020884,-2.026824,-0.368426
27,model_arch1_r0.9_Ld0.7_Lp0.3_seed8561,[1],-0.177143,-0.020884,-2.026824,-0.368426
326,model_arch11_r0.9_Ld0.7_Lp0.3_seed3211,[11],-0.127772,0.193254,-2.289943,-0.393465
58,model_arch2_r0.9_Ld0.7_Lp0.3_seed2023,[2],-0.150708,0.080838,-2.417051,-0.398257
53,model_arch2_r0.01_Ld0.7_Lp0.3_seed2023,[2],-0.154884,0.112705,-2.497212,-0.407044



📊 MÉTRICAS COMPLETAS - TOP 5 (y)


,model,Neurons,R2_ZZx1_y,R2_ZZx2_y,R2_ZZxReto_y,R2_ZZy1_y,R2_ZZy2_y,R2_LSG_1_y,R2_LSG_2_y,R2_ZZx1_inv_y,R2_semiCirc_y,R2_train_mean,R2_val_mean,R2_test_mean,Score
22,model_arch1_r0.01_Ld0.7_Lp0.3_seed8561,[1],-0.177143,-0.020884,0.804202,-4.448029,-8.668298,0.049608,-2.044745,-0.121143,0.240637,-0.177143,-0.020884,-2.026824,-0.368426
27,model_arch1_r0.9_Ld0.7_Lp0.3_seed8561,[1],-0.177143,-0.020884,0.804202,-4.448029,-8.668298,0.049608,-2.044745,-0.121143,0.240637,-0.177143,-0.020884,-2.026824,-0.368426
326,model_arch11_r0.9_Ld0.7_Lp0.3_seed3211,[11],-0.127772,0.193254,0.794814,-2.712286,-10.210667,0.608950,-3.669918,-0.946796,0.106300,-0.127772,0.193254,-2.289943,-0.393465
58,model_arch2_r0.9_Ld0.7_Lp0.3_seed2023,[2],-0.150708,0.080838,0.786297,-5.342112,-9.746146,-0.328703,-2.178655,-0.380478,0.270437,-0.150708,0.080838,-2.417051,-0.398257
53,model_arch2_r0.01_Ld0.7_Lp0.3_seed2023,[2],-0.154884,0.112705,0.778951,-5.498393,-9.951154,-0.409504,-2.263780,-0.431044,0.294441,-0.154884,0.112705,-2.497212,-0.407044


In [5]:
final_table.to_excel("BestModels-otm.xlsx")